In [53]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

In [54]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [55]:
text = Path("tiny-shakespeare.txt").read_text()

In [56]:
print(text[:500])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [57]:
class Tokenizer:
    def __init__(self,vocab):
        super().__init__()
        self.token_2_id = {
            char:i for i,char in enumerate(vocab)
        }
        self.id_2_token = {
            i:char for i,char in enumerate(vocab)
        }
    @staticmethod
    def train(raw_text):
        vocab = set(text)
        return Tokenizer(vocab)
    
    def encode(self,prompt):
        ret = []
        for x in prompt:
            ret.append(self.token_2_id[x])
        return torch.tensor(ret,dtype = torch.long)
    
    def decode(self,token_ids):
        ret = [] 
        for id in token_ids:
            ret.append(self.id_2_token[int(id)])
        return "".join(ret)
    
    def size_of_vocab(self):
        return len(self.token_2_id)


### Test tokenizer

In [58]:
tokenizer = Tokenizer.train(text)
print(tokenizer.size_of_vocab())

65


In [59]:
tokenizer.decode(tokenizer.encode("check this out"))

'check this out'

### Prepare dataset

In [60]:
from torch.utils.data import Dataset,DataLoader

In [61]:
class Ds(Dataset):
    def __init__(self,data,block_len):
        self.data = data
        self.block_size = block_len
    
    def __len__(self):
        return len(self.data)-self.block_size + 1
    
    def __getitem__(self, index):
        assert index < len(self.data)-self.block_size
        X = self.data[index:index+self.block_size]
        Y = self.data[index+1:index+self.block_size+1]
        return X,Y


### Attention head

In [62]:
config = {
    "vocabulary_size": tokenizer.size_of_vocab(),
    "context_size": 256,
    "d_embed": 768,
    "heads_num": 12,
    "layers_num": 6,
    "dropout_rate": 0.1,
    "use_bias": False,
    "batch":32
}

config["head_size"] = config["d_embed"] // config["heads_num"]

In [63]:
class SingleHead(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.Q_W = nn.Linear(config['d_embed'],config["head_size"])
        self.K_W = nn.Linear(config['d_embed'],config["head_size"])
        self.V_W = nn.Linear(config['d_embed'],config["head_size"])
        self.mask = torch.tril(torch.ones(config['context_size'],config['context_size']))
    
    def forward(self,x):
        # x.shape [B,T,D_E]
        q = self.Q_W(x)  # size : [B,T,HS]
        k = self.Q_W(x)
        v = self.Q_W(x)
        
        q_kT = q@k.transpose(1,2)  # shape [B,T,T]
        q_kT_masked = q_kT.masked_fill(self.mask[:x.shape[1],:x.shape[1]]==0,-torch.inf)
        scaled_q_kT = q_kT_masked/k.shape[-1]**0.5
        softmax_scores = F.softmax(scaled_q_kT,dim=-1) # dim-1 => accross keys or columns
        values = softmax_scores @ v
        return values

### test singl head

In [64]:
head = SingleHead(config)
randomInp  = torch.randn(config['batch'],config['context_size'],config['d_embed'])
out = head(randomInp)
print(out.shape)

torch.Size([32, 256, 64])


# MHA

In [71]:
class MHA(nn.Module):
    def __init__(self,config):
        super().__init__()
        heads = [SingleHead(config) for _ in range(config['heads_num'])]
        self.mha = nn.ModuleList(heads)
        self.out_proj = nn.Linear(config['d_embed'],config['d_embed'])
    
    def forward(self,x):
        outputs = [head(x) for head in self.mha]
        #print(len(outputs),type(outputs),type(outputs[0]))
        output = torch.cat(outputs,dim=-1)
        return self.out_proj(output)


test single mha

In [72]:
mha = MHA(config)
randomInp  = torch.randn(config['batch'],config['context_size'],config['d_embed'])
out = mha(randomInp)
print(out.shape)

torch.Size([32, 256, 768])


# FFN in a block

In [73]:
class FFN(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.ffn = nn.Sequential(
            nn.Linear(config['d_embed'],4*config['d_embed']),
            nn.GELU(),
            nn.Linear(4*config['d_embed'],config['d_embed'])
        )
    def forward(self,x):
        return self.ffn(x)

In [75]:
test_ffn = FFN(config)
rand_input = torch.rand(config['batch'], config["context_size"], config["d_embed"])
FFN(config)(rand_input).shape

torch.Size([32, 256, 768])

# TRansformer Block

In [76]:
nn.LayerNorm?

Init signature:
nn.LayerNorm(
    normalized_shape: Union[int, list[int], torch.Size],
    eps: float = 1e-05,
    elementwise_affine: bool = True,
    bias: bool = True,
    device=None,
    dtype=None,
) -> None
Docstring:     
Applies Layer Normalization over a mini-batch of inputs.

This layer implements the operation as described in
the paper `Layer Normalization <https://arxiv.org/abs/1607.06450>`__

.. math::
    y = \frac{x - \mathrm{E}[x]}{ \sqrt{\mathrm{Var}[x] + \epsilon}} * \gamma + \beta

The mean and standard-deviation are calculated over the last `D` dimensions, where `D`
is the dimension of :attr:`normalized_shape`. For example, if :attr:`normalized_shape`
is ``(3, 5)`` (a 2-dimensional shape), the mean and standard-deviation are computed over
the last 2 dimensions of the input (i.e. ``input.mean((-2, -1))``).
:math:`\gamma` and :math:`\beta` are learnable affine transform parameters of
:attr:`normalized_shape` if :attr:`elementwise_affine` is ``True``.
The variance is 

In [79]:
class Block(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.mha = MHA(config)
        self.LN1 = nn.LayerNorm(config['d_embed'])
        self.LN2 = nn.LayerNorm(config['d_embed'])
        self.ffn = FFN(config)
    
    def forward(self,x):
        residue = x
        x = self.mha(self.LN1(x))
        x = x + residue
        residue = x
        x = self.ffn(self.LN2(x))
        return x + residue

In [80]:
transformer_block = Block(config)
transformer_block(randomInp).shape

torch.Size([32, 256, 768])

In [83]:
class GPT(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.pos_embedding = nn.Embedding(config['context_size'],config['d_embed'])
        self.token_embedding = nn.Embedding(config['vocabulary_size'],config['d_embed'])
        
        blocks = [Block(config) for _ in range(config['layers_num'])]
        
        self.transformer_blocks = nn.Sequential(*blocks)
        self.output_proj = nn.Linear(config['d_embed'],config['vocabulary_size'],bias=False)
        self.layer_norm = nn.LayerNorm(config["d_embed"])
        
    def forward(self,token_ids):
        embed = self.token_embedding(token_ids)
        sequence = torch.arange(token_ids.shape[-1], device=device)
        embed = embed + self.pos_embedding(sequence)
        
        x = self.transformer_blocks(embed)
        x = self.layer_norm(x)
        return self.output_proj(x)

In [85]:
model = GPT(config).to(device)
output = model(tokenizer.encode("Hi").unsqueeze(dim=0).to(device))

In [87]:
output.shape

torch.Size([1, 2, 65])

# Try & generate o/p

In [91]:
def generate_output(model, prompt_ids, max_tokens):
    output_ids = prompt_ids
    if output_ids.shape[1] >= config["context_size"]:
        return
    for _ in range(max_tokens):
        with torch.no_grad():
            logits = model(output_ids)

        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        # Sample a random token
        next_token_id = torch.multinomial(probs, num_samples=1)
        output_ids = torch.cat([output_ids, next_token_id], dim=-1)
    print(output_ids.shape)
    return output_ids

In [98]:
def generate_with_prompt(model, tokenizer, prompt, max_tokens=100):
    model.eval()

    prompt = tokenizer.encode(prompt).unsqueeze(dim=0).to(device)
    out = generate_output(model, prompt, max_tokens=max_tokens)
    return tokenizer.decode(out.squeeze())

In [99]:
generate_with_prompt(model, tokenizer, "First Citizen:\n")

torch.Size([1, 115])


"First Citizen:\ng?\n;VTVEHx&OnMKC.3WmuCAA3twWpBDt Nj?nnQrJ ,EZrE.maU:wcbBWx;g'loz3HUT?skFGHNj'qq!c-cfly-hT!kEEaqD?flT"

## TODO implement training